Zoom: The key idea is **not to zoom the graph by repeatedly resizing a Pygame `Surface`**. Instead, keep your graph in a logical coordinate system and apply a **camera transformation** when drawing it.

 Pygame doesn't have a built-in "zoom camera" because this is normally implemented with coordinates and transformations. `pygame.transform.scale()` exists, but it is primarily for resizing surfaces/images.  Pygame

 ## The basic idea

 Suppose your graph has nodes at logical positions:

```python
nodes = {
    "A": (100, 100),
    "B": (300, 150),
    "C": (200, 400),
}
```

 Keep those positions unchanged.

 Then maintain:

```python
zoom = 1.0
camera_x = 0
camera_y = 0
```

 When you draw something, convert its **world coordinates** to **screen coordinates**:

```python
screen_x = (world_x - camera_x) * zoom
screen_y = (world_y - camera_y) * zoom
```

 So if:

```python
world position = (300, 150)
zoom = 2
camera = (100, 50)
```

 you get:

```python
screen position = ((300 - 100) * 2,
                   (150 - 50) * 2)

                 = (400, 200)
```

 This is essentially a 2D camera.

 ## A small Pygame example

 Here's a complete minimal example:

```python
import pygame

pygame.init()

WIDTH, HEIGHT = 1000, 700
screen = pygame.display.set_mode((WIDTH, HEIGHT))
clock = pygame.time.Clock()

# Graph coordinates -- these never change.
nodes = {
    "A": (100, 100),
    "B": (400, 150),
    "C": (250, 400),
    "D": (600, 350),
}

edges = [
    ("A", "B"),
    ("A", "C"),
    ("B", "D"),
    ("C", "D"),
]

zoom = 1.0
camera_x = 0
camera_y = 0

running = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        # Mouse wheel = zoom
        elif event.type == pygame.MOUSEWHEEL:
            if event.y > 0:
                zoom *= 1.1
            elif event.y < 0:
                zoom /= 1.1

            # Prevent extreme zoom levels
            zoom = max(0.2, min(zoom, 5.0))

    screen.fill((30, 30, 30))

    # Draw edges
    for a, b in edges:
        ax, ay = nodes[a]
        bx, by = nodes[b]

        ax = (ax - camera_x) * zoom
        ay = (ay - camera_y) * zoom

        bx = (bx - camera_x) * zoom
        by = (by - camera_y) * zoom

        pygame.draw.line(
            screen,
            (180, 180, 180),
            (ax, ay),
            (bx, by),
            max(1, int(3 * zoom))
        )

    # Draw nodes
    for name, (x, y) in nodes.items():
        screen_x = (x - camera_x) * zoom
        screen_y = (y - camera_y) * zoom

        radius = max(3, int(15 * zoom))

        pygame.draw.circle(
            screen,
            (50, 150, 255),
            (int(screen_x), int(screen_y)),
            radius
        )

    pygame.display.flip()
    clock.tick(60)

pygame.quit()
```

 Now the entire graph zooms in and out while the underlying graph data remains exactly the same.

 ## Even better: zoom toward the mouse

 A nice graph viewer usually zooms **toward the mouse cursor**, rather than toward the top-left corner.

 For that, you need to make sure the world coordinate underneath the mouse stays underneath the mouse after zooming.

 The important part is:

```python
mouse_x, mouse_y = pygame.mouse.get_pos()

# World coordinate currently underneath mouse
world_x = mouse_x / zoom + camera_x
world_y = mouse_y / zoom + camera_y

# Change zoom
zoom *= 1.1

# Adjust camera so the same world position stays under mouse
camera_x = world_x - mouse_x / zoom
camera_y = world_y - mouse_y / zoom
```

 So your wheel handler can become:

```python
elif event.type == pygame.MOUSEWHEEL:
    mouse_x, mouse_y = pygame.mouse.get_pos()

    # Position in the graph under the cursor
    world_x = mouse_x / zoom + camera_x
    world_y = mouse_y / zoom + camera_y

    if event.y > 0:
        zoom *= 1.1
    else:
        zoom /= 1.1

    zoom = max(0.2, min(zoom, 5.0))

    # Keep that graph position under the cursor
    camera_x = world_x - mouse_x / zoom
    camera_y = world_y - mouse_y / zoom
```

 This gives you the familiar behavior of zooming in on whatever part of the graph you're pointing at.

 ## Adding panning

 For a graph project, I'd also implement mouse-drag panning. For example:

```python
dragging = False
last_mouse = None
```

 Then:

```python
for event in pygame.event.get():

    if event.type == pygame.MOUSEBUTTONDOWN:
        if event.button == 1:
            dragging = True
            last_mouse = pygame.mouse.get_pos()

    elif event.type == pygame.MOUSEBUTTONUP:
        if event.button == 1:
            dragging = False

    elif event.type == pygame.MOUSEMOTION:
        if dragging:
            mouse_x, mouse_y = event.pos

            dx = mouse_x - last_mouse[0]
            dy = mouse_y - last_mouse[1]

            camera_x -= dx / zoom
            camera_y -= dy / zoom

            last_mouse = (mouse_x, mouse_y)
```

 Notice the `/ zoom`. That's important: when you're zoomed in, dragging 100 screen pixels shouldn't move the graph by 100 logical pixels.

 ## One important design decision

 I'd recommend structuring your program into three layers:

```python
Graph data
    ↓
World coordinates
    ↓
Camera (zoom + position)
    ↓
Screen coordinates
    ↓
Pygame drawing
```

 For example:

```python
def world_to_screen(x, y):
    return (
        (x - camera_x) * zoom,
        (y - camera_y) * zoom
    )
```

 Then your drawing code becomes much cleaner:

```python
x, y = world_to_screen(*nodes["A"])

pygame.draw.circle(
    screen,
    BLUE,
    (int(x), int(y)),
    int(15 * zoom)
)
```

 And eventually you can also implement the inverse:

```python
def screen_to_world(x, y):
    return (
        x / zoom + camera_x,
        y / zoom + camera_y
    )
```

 That second function is particularly useful when the user clicks a node: the mouse gives you a **screen coordinate**, but your graph needs to know which node is at the corresponding **world coordinate**.

 ### One more thing

 For a graph with potentially thousands of nodes, **don't create a gigantic Surface containing the whole graph and then scale it**. Draw the graph from its logical coordinates using the camera transformation. Pygame's `Surface` has a fixed resolution, while transforms create resized surfaces, so treating the graph itself as a huge bitmap isn't a good fit for an arbitrarily large graph.  Pygame+1

 This approach also makes **zoom, pan, node selection, dragging nodes, grid drawing, and eventually culling off-screen nodes** much easier to implement.